In [12]:
# Sieć neuronowa rozpoznająca prawdziwość twarzy
# SPRÓBUJEMY WYBRAĆ JAKO WEJŚCIA TYLKO OTOCZENIE TWARZY, A NIE CAŁOŚĆ ZDJĘCIA
# Fragmenty oznaczone jako [PARAMETR] mogą być modyfikowane w celu uzyskania lepszych wyników
import os
import cv2

# Wczytanie danych
original_faces_path = "../../../Dane/NN raw/original"
original_faces = []

spoof_faces_path = "../../../Dane/NN raw/spoof"
spoof_faces = []

# Wczytanie oryginalnych twarzy
files = [f for f in os.listdir(original_faces_path) if f.endswith(".jpg") or f.endswith(".png") or f.endswith(".webp")] # Tylko pliki JPG, PNG i WEBP
for f in files:
    image = cv2.imread(os.path.join(original_faces_path, f))
    original_faces.append(image)

# Wczytanie podstawionych twarzy
files = [f for f in os.listdir(spoof_faces_path) if f.endswith(".jpg") or f.endswith(".png") or f.endswith(".webp")] # Tylko pliki JPG, PNG i WEBP
for f in files:
    image = cv2.imread(os.path.join(spoof_faces_path, f))
    spoof_faces.append(image)

print(f"Liczba oryginalnych twarzy: {len(original_faces)}")
print(f"Liczba podstawionych twarzy: {len(spoof_faces)}")

Liczba oryginalnych twarzy: 142
Liczba podstawionych twarzy: 119


In [13]:
# Data augmentation
from torchvision import transforms
import numpy as np

# Utworzenie generatora przerabiającego obrazy
# [PARAMETRY]
aug = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),  # Losowe odbicie lustrzane
    transforms.RandomRotation(5),  # Losowe obrócenie o kąt z przedziału [-5, 5]
    transforms.ColorJitter(brightness=0.1, saturation=0.1),  # Zmiana jasności i nasycenia
])

# Wygenerowanie i zapisanie nowych obrazów
original_faces_aug = []
spoof_faces_aug = []

for face in original_faces:
    for i in range(3):
        original_faces_aug.append(np.array(aug(face)))
        
for face in spoof_faces:
    for i in range(3):
        spoof_faces_aug.append(np.array(aug(face)))

In [14]:
# Wstępne przetwarzanie danych
import torch
import cv2.data
from sklearn.preprocessing import LabelEncoder

face_classifier = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Wykrywanie twarzy, rozszerzenie tego obszaru o 33% w każdą stronę
# Następnie przeskalowanie do (64, 64) i zapisanie
def get_image_part(img_p):
    faces = face_classifier.detectMultiScale(img_p, 1.1, 5)
    if len(faces) == 0:
        return None
    (x, y, w, h) = faces[0]
    x = max(0, x - w // 3)
    y = max(0, y - h // 3)
    w = min(img_p.shape[1], w * 5 // 3)
    h = min(img_p.shape[0], h * 5 // 3)
    return img_p[y:y+h, x:x+w]


original_nn_path = "../../../Dane/NN/original"
spoof_nn_path = "../../../Dane/NN/spoof"

# Przetwarzanie i zapisywanie obrazów
i = 0
for img in original_faces_aug:
    img_part = get_image_part(img)
    if img_part is not None:
        cv2.imwrite(os.path.join(original_nn_path, f"{i}.png"), img_part)
        i += 1
        
i = 0
for img in spoof_faces_aug:
    img_part = get_image_part(img)
    if img_part is not None:
        cv2.imwrite(os.path.join(spoof_nn_path, f"{i}.png"), img_part)
        i += 1

In [15]:
# Wczytanie nowych danych
original_faces_resized = []
spoof_faces_resized = []

face_size = (64, 64)

# Wczytanie oryginalnych twarzy
files = [f for f in os.listdir(original_nn_path) if f.endswith(".png")] # Tylko pliki PNG
for f in files:
    image = cv2.imread(os.path.join(original_nn_path, f))
    original_faces_resized.append(cv2.resize(image, face_size))
    
# Wczytanie podstawionych twarzy
files = [f for f in os.listdir(spoof_nn_path) if f.endswith(".png")] # Tylko pliki PNG
for f in files:
    image = cv2.imread(os.path.join(spoof_nn_path, f))
    spoof_faces_resized.append(cv2.resize(image, face_size))

# Zamiana obrazów na macierze zawierające wartości z przedziału [0, 1] zamiast [0, 255]
original_faces_normalized = np.array(original_faces_resized) / 255.0
spoof_faces_normalized = np.array(spoof_faces_resized) / 255.0

# Utworzenie kategorii, do których sieć będzie przydzielać obrazy
labels = np.array(["original"] * len(original_faces_normalized) + ["spoof"] * len(spoof_faces_normalized))
le = LabelEncoder().fit(labels)
labels_encoded = le.transform(labels)
labels_encoded = torch.tensor(labels_encoded, dtype=torch.int64)

print(labels_encoded)

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [16]:
# Przerabianie obrazów
# Utworzenie generatora przerabiającego obrazy
# Obrazy są już przerobione wyżej, więc tylko konwersja na tensor.
aug = transforms.Compose([
    transforms.ToTensor()
])

In [17]:
# Utworzenie zbiorów danych
from sklearn.model_selection import train_test_split

# Połączenie zbiorów danych z odpowiednimi kategoriami
original_faces_tensor = torch.stack([aug(face) for face in original_faces_normalized])
spoof_faces_tensor = torch.stack([aug(face) for face in spoof_faces_normalized])
faces_all = np.concatenate((original_faces_tensor, spoof_faces_tensor))

# Podział danych na zbiór treningowy i testowy
(trainX, testX, trainY, testY) = train_test_split(faces_all, labels_encoded, test_size=0.25, random_state=61185)

# Konwersja danych na tensory
trainX = torch.tensor(trainX, dtype=torch.float32)
testX = torch.tensor(testX, dtype=torch.float32)
trainY = torch.tensor(trainY, dtype=torch.long)
testY = torch.tensor(testY, dtype=torch.long)

# Wyświetlenie części zbiorów
print(trainX)

tensor([[[[0.6706, 0.6706, 0.6706,  ..., 0.6863, 0.6863, 0.6863],
          [0.6706, 0.6706, 0.6706,  ..., 0.6863, 0.6863, 0.6863],
          [0.6706, 0.6706, 0.6706,  ..., 0.6824, 0.6745, 0.6745],
          ...,
          [0.8078, 0.8627, 0.7569,  ..., 0.5373, 0.5490, 0.5490],
          [0.7490, 0.7020, 0.6902,  ..., 0.8784, 0.8510, 0.5059],
          [0.7294, 0.7647, 0.7451,  ..., 0.8471, 0.8314, 0.8275]],

         [[0.6863, 0.6863, 0.6863,  ..., 0.7020, 0.7020, 0.7020],
          [0.6863, 0.6863, 0.6863,  ..., 0.7020, 0.7020, 0.6980],
          [0.6863, 0.6863, 0.6863,  ..., 0.6941, 0.6902, 0.6902],
          ...,
          [0.8039, 0.8588, 0.7451,  ..., 0.5373, 0.5490, 0.5490],
          [0.7412, 0.6980, 0.6784,  ..., 0.8784, 0.8549, 0.5059],
          [0.7176, 0.7529, 0.7412,  ..., 0.8471, 0.8314, 0.8314]],

         [[0.6824, 0.6824, 0.6824,  ..., 0.6980, 0.6980, 0.6980],
          [0.6824, 0.6824, 0.6824,  ..., 0.6980, 0.6980, 0.6980],
          [0.6824, 0.6824, 0.6824,  ..., 0

/var/folders/42/03wqtylj4nz_4lpvj_w6mc2h0000gn/T/ipykernel_65298/2649742169.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  trainY = torch.tensor(trainY, dtype=torch.long)
/var/folders/42/03wqtylj4nz_4lpvj_w6mc2h0000gn/T/ipykernel_65298/2649742169.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  testY = torch.tensor(testY, dtype=torch.long)


In [18]:
# Utworzenie modelu sieci neuronowej
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

# Model sieci neuronowej
class SpoofDetectionNet(nn.Module):
    def __init__(self):
        super(SpoofDetectionNet, self).__init__()
        # CONV => RELU => CONV => RELU => POOL
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding='same')
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, padding='same')
        self.bn2 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.dropout1 = nn.Dropout(0.25)

        # CONV => RELU => CONV => RELU => POOL
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding='same')
        self.bn3 = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32, 32, kernel_size=3, padding='same')
        self.bn4 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        self.dropout2 = nn.Dropout(0.25)

        # FC => RELU
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 16 * 16, 64)
        self.bn5 = nn.BatchNorm1d(64)
        self.dropout3 = nn.Dropout(0.5)

        # Warstwa wyjściowa
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        # CONV => RELU => CONV => RELU => POOL
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.dropout1(x)

        # CONV => RELU => CONV => RELU => POOL
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.dropout2(x)

        # FC => RELU
        x = self.flatten(x)
        x = F.relu(self.bn5(self.fc1(x)))
        x = self.dropout3(x)

        # Klasyfikator Softmax
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [21]:
# Utworzenie modelu
model = SpoofDetectionNet()
criterion = nn.CrossEntropyLoss()

# Utworzenie optimizera
# [PARAMETRY]
INIT_LR = 1e-3
EPOCHS = 10
optimizer = Adam(model.parameters(), lr=INIT_LR, weight_decay=INIT_LR / EPOCHS)

In [22]:
# Trenowanie modelu
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

# Przygotowanie danych
train_dataset = TensorDataset(trainX, trainY)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(testX, testY)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Utworzenie obiektu do zapisywania logów z trenowania modelu
writer = SummaryWriter()

# Pętla trenująca model
model.train()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for epoch in range(EPOCHS):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")
    writer.add_scalar('training loss', running_loss / len(train_loader), epoch)

    # Validation loss
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_loss /= len(test_loader)
    val_accuracy = 100 * correct / total
    print(f"Validation Loss: {val_loss}, Accuracy: {val_accuracy}%")
    writer.add_scalar('validation loss', val_loss, epoch)
    writer.add_scalar('validation accuracy', val_accuracy, epoch)

writer.close()

Epoch 1, Loss: 0.5884910576483783
Validation Loss: 0.6632269223531088, Accuracy: 65.74585635359117%
Epoch 2, Loss: 0.6153077693546519
Validation Loss: 0.6151378452777863, Accuracy: 65.74585635359117%
Epoch 3, Loss: 0.46329359271947074
Validation Loss: 0.4653131415446599, Accuracy: 75.13812154696133%
Epoch 4, Loss: 0.41869334350613985
Validation Loss: 0.5111655096213022, Accuracy: 75.69060773480663%
Epoch 5, Loss: 0.41624662104774923
Validation Loss: 0.42271678149700165, Accuracy: 81.21546961325967%
Epoch 6, Loss: 0.3029366310905008
Validation Loss: 0.38057095805803937, Accuracy: 82.32044198895028%
Epoch 7, Loss: 0.20701413601636887
Validation Loss: 0.36832696323593456, Accuracy: 86.74033149171271%
Epoch 8, Loss: 0.13606857771382613
Validation Loss: 0.40070494016011554, Accuracy: 85.6353591160221%
Epoch 9, Loss: 0.14303205118459814
Validation Loss: 0.43427998572587967, Accuracy: 86.74033149171271%
Epoch 10, Loss: 0.12559212908587036
Validation Loss: 0.5253873318433762, Accuracy: 86.7403

In [23]:
# Wypisanie wytrenowanych wag modelu
print("Model's state_dict:")
for param_tensor in model.state_dict():
    print(param_tensor, "\t", model.state_dict()[param_tensor].size())

Model's state_dict:
conv1.weight 	 torch.Size([16, 3, 3, 3])
conv1.bias 	 torch.Size([16])
bn1.weight 	 torch.Size([16])
bn1.bias 	 torch.Size([16])
bn1.running_mean 	 torch.Size([16])
bn1.running_var 	 torch.Size([16])
bn1.num_batches_tracked 	 torch.Size([])
conv2.weight 	 torch.Size([16, 16, 3, 3])
conv2.bias 	 torch.Size([16])
bn2.weight 	 torch.Size([16])
bn2.bias 	 torch.Size([16])
bn2.running_mean 	 torch.Size([16])
bn2.running_var 	 torch.Size([16])
bn2.num_batches_tracked 	 torch.Size([])
conv3.weight 	 torch.Size([32, 16, 3, 3])
conv3.bias 	 torch.Size([32])
bn3.weight 	 torch.Size([32])
bn3.bias 	 torch.Size([32])
bn3.running_mean 	 torch.Size([32])
bn3.running_var 	 torch.Size([32])
bn3.num_batches_tracked 	 torch.Size([])
conv4.weight 	 torch.Size([32, 32, 3, 3])
conv4.bias 	 torch.Size([32])
bn4.weight 	 torch.Size([32])
bn4.bias 	 torch.Size([32])
bn4.running_mean 	 torch.Size([32])
bn4.running_var 	 torch.Size([32])
bn4.num_batches_tracked 	 torch.Size([])
fc1.weight 	 

In [24]:
# Zapisanie modelu
torch.save(model.state_dict(), "model_closeup.pth")
torch.save(model, "model_closeup_full.pth")